### Database Query Engine START

In [1]:
# Import the required packages
import os
from os.path import dirname as one_folder_up, join as join_paths
import sqlite3

# Import pandas to read query outputs as dfs, read source files, numpy 
import pandas as pd
import numpy as np

from datetime import datetime

import traceback

C:\Users\tatil\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\tatil\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [2]:
# Define the functions

# START - DB
# Connects to an sqlite db and returns the conn object
def connect_to_database(db_path):
    try:
        conn = sqlite3.connect(db_path)
        return conn
    except Exception as e:
        return False

# Execute a select query against a DB
def execute_db_select(conn, query):
    try:
        df = pd.read_sql_query(query, conn)
        return df
    except Exception as e:
        print(traceback.format_exc())
        return False

# Gets an insert query along with a pandas dataframe, inserts the dataframe into the DB  
def execute_db_insert(conn, query, data):
    try:
        # Create cursor, execute the query, commit, then close the cursor
        cur = conn.cursor()
        cur.executemany(query, data)
        conn.commit()
        cur.close()
        return True
    except Exception as e:
        return False

# Deletes rows from a table based on an array of ids for a particular column
def execute_db_delete(conn, query, ids):
    try:
        # Create cursor, execute the query, commit, then close the cursor
        cur = conn.cursor()
        cur.executemany(query, ids)
        conn.commit()
        cur.close()
        return True
    except Exception as e:
        return False

def execute_db_query(conn, query):
    try:
        # Create cursor, execute the query, commit, then close the cursor
        cur = conn.cursor()
        cur.execute(query)
        conn.commit()
        cur.close()
        return True
    except Exception as e:
        return False
# END - DB

In [3]:
# Define parameters
target_db               = "courses.db"
target_table            = "enrollments"

In [4]:
# Derive required variable(s)
current_folder          = os.getcwd()
root_folder             = one_folder_up(current_folder) 
""" 
    As the current file sits here "\gic_assignment\1_Data_Ingestion_Pipeline\"
    and the root folder is gic_assignment. 
"""

data_folder             = join_paths(root_folder, "data")
output_folder           = join_paths(data_folder, "output")

target_db_path          = join_paths(output_folder, target_db)

### Database Query Engine END

## SQL REPORTING Statement

### Show the current state of the table

In [5]:
conn = connect_to_database(target_db_path)
results = execute_db_select(conn, "SELECT * FROM enrollments")
print(results)
conn.close()

    enrollment_id participant_id participant_name  course_id course_date  \
0               1           P001        Alice Tan          1  2024-01-15   
1               2           P002          Bob Lee          1  2024-01-15   
2               3           P007        Grace Koh          1  2024-01-20   
3               4           P009       Irene Chua          1  2024-01-25   
4               5           P003       Charlie Ng          2  2024-02-01   
5               6           P001        Alice Tan          2  2024-02-01   
6               7           P004        Diana Lim          1  2024-02-10   
7               8           P007        Grace Koh          2  2024-02-15   
8               9           P009       Irene Chua          2  2024-02-20   
9              10           P002          Bob Lee          3  2024-03-05   
10             11           P005       Edward Goh          4  2024-03-15   
11             12           P003       Charlie Ng          1  2024-03-20   
12          

#### Assumptions
- As we only have one date column available in the table, i.e.,. course_date, we have to assume that this is the date on which the participant joined the course. We can also infer this by looking at the course_dates for the course_id 1 when Alice Tan (2024-01-15) and Grace Koh (2024-01-20) enrolled
- As we don't have a course completion date, we have to assume that the enrollee will be able to complete a given course in 8 weeks (56 days)
  - Edit: Updated the threshold from 1 week to 8 weeks as I did not find any participants who did not meet the requirements

In [96]:
# Set the threshold
no_of_days_to_autocomplete_course = 56

# Execute the query
conn = connect_to_database(target_db_path)
reporting_query = f"""
    WITH RECURSIVE courses_with_prerequisites AS (
        SELECT course_id,
            SUBSTR(prerequisites, 1, INSTR(prerequisites, ',') - 1) AS prerequisite,
            SUBSTR(prerequisites, INSTR(prerequisites, ',') + 1) AS remaining_prerequisites
        FROM
        (
            SELECT course_id,
                REPLACE(REPLACE(REPLACE(prerequisites, '[', ''), ']', ''), ' ', '')  || ',' AS prerequisites
            FROM courses
            WHERE prerequisites IS NOT NULL
        ) AS courses_base

        UNION ALL

        SELECT course_id,
            SUBSTR(remaining_prerequisites, 1, INSTR(remaining_prerequisites, ',') - 1) AS prerequisite,
            SUBSTR(remaining_prerequisites, INSTR(remaining_prerequisites, ',') + 1) AS remaining_prerequisites
        FROM courses_with_prerequisites
        WHERE remaining_prerequisites != ''
    ),

    courses_with_prerequisites_no_nulls AS (
        SELECT course_id,
            prerequisite
        FROM courses_with_prerequisites
        WHERE prerequisite != ''
        AND prerequisite IS NOT NULL
        GROUP BY course_id,
            prerequisite
    ),

    courses_with_prerequisite_list AS (
        SELECT course_id,
            GROUP_CONCAT(prerequisite, ',') AS prerequisite_list
        FROM courses_with_prerequisites_no_nulls
        GROUP BY course_id
    ),

    courses_with_prerequisite_counts AS (
        SELECT base.course_id,
            base.prerequisite,
            counts.total_prerequisites
        FROM courses_with_prerequisites_no_nulls base
        INNER JOIN
        (
            SELECT course_id,
                COUNT(prerequisite) AS total_prerequisites
            FROM courses_with_prerequisites_no_nulls
            GROUP BY course_id
        ) counts
        ON base.course_id = counts.course_id
    ),

    enrollments_base AS (
        SELECT participant_id,
            participant_name,
            course_id,
            course_date
        FROM enrollments
    ),

    enrollments_courses AS (
        SELECT E.participant_id,
            E.participant_name,
            E.course_id,
            E.course_date,
            COALESCE(C.total_prerequisites, 0) AS total_prerequisites,
            C.prerequisite AS course_id_prerequisite
        FROM enrollments_base E
        LEFT JOIN courses_with_prerequisite_counts C
        ON E.course_id = C.course_id
    ),

    enrollments_with_earliest_enrollment AS (
        SELECT participant_id,
            course_id,
            course_date
        FROM
        (
            SELECT participant_id,
                course_id,
                course_date,
                ROW_NUMBER() OVER(PARTITION BY participant_id, course_id ORDER BY course_date) AS enrollment_order
            FROM enrollments_base
        ) AS enrollments_with_rn
        WHERE enrollment_order = 1
    ),

    enrollments_courses_prerequisite_status AS (
        SELECT EnrollmentsCourses.*,
            PrerequisiteCourse.course_date as course_date_prerequisite,
            CASE 
                WHEN PrerequisiteCourse.course_date IS NULL
                THEN NULL
                WHEN (STRFTIME('%s', EnrollmentsCourses.course_date) - STRFTIME('%s', PrerequisiteCourse.course_date)) / 86400 >= {no_of_days_to_autocomplete_course}
                THEN 1
                ELSE 0
            END AS prerequisite_completion_status
        FROM enrollments_courses AS EnrollmentsCourses
        LEFT JOIN enrollments_with_earliest_enrollment AS PrerequisiteCourse
        ON EnrollmentsCourses.participant_id = PrerequisiteCourse.participant_id
        AND EnrollmentsCourses.course_id_prerequisite = PrerequisiteCourse.course_id
    ),

    enrollments_courses_completed_prerequisites AS (
        SELECT base.*,
            completed.prerequisites_completed,
            CASE 
                WHEN base.total_prerequisites = 0
                OR base.total_prerequisites = completed.prerequisites_completed
                THEN 'Yes'
                WHEN base.total_prerequisites > completed.prerequisites_completed
                THEN 'No'
                WHEN completed.prerequisites_completed IS NULL
                THEN 'No'
            END AS has_completed_all_the_prerequisites
        FROM enrollments_courses_prerequisite_status base 
        LEFT JOIN (
            SELECT participant_id,
                course_id,
                course_date,
                COALESCE(COUNT(DISTINCT course_id_prerequisite), 0) AS prerequisites_completed
            FROM enrollments_courses_prerequisite_status
            WHERE prerequisite_completion_status = 1
            GROUP BY participant_id,
                course_id,
                course_date
        ) completed
        ON base.participant_id = completed.participant_id
        AND base.course_id = completed.course_id
        AND base.course_date = completed.course_date
    ),

    enrollments_with_unmet_prerequisites AS (
        SELECT 
            participant_id,
            participant_name,
            course_id,
            course_date,
            total_prerequisites AS prereq_to_complete,
            coalesce(prerequisites_completed, 0) AS prereq_completed,
            GROUP_CONCAT(course_id_prerequisite, ',') AS prereq_to_complete_list,
            COALESCE(
                GROUP_CONCAT(
                    CASE
                        WHEN prerequisite_completion_status = 1
                        THEN course_id_prerequisite
                    END, 
                    ','
                ), ''
            )AS prereq_completed_list
        FROM enrollments_courses_completed_prerequisites
        WHERE has_completed_all_the_prerequisites = 'No'
        GROUP BY participant_id,
            participant_name,
            course_id,
            course_date,
            total_prerequisites,
            coalesce(prerequisites_completed, 0)
    )

    SELECT participant_id,
        participant_name,
        course_id,
        course_date,
        prereq_to_complete || ' [' || prereq_to_complete_list || ']' AS prereq_to_complete,
        prereq_completed || ' [' || prereq_completed_list || ']' AS prereq_completed
    FROM enrollments_with_unmet_prerequisites
    ;
"""
results = execute_db_select(conn, reporting_query)
print(results.to_markdown(index = False))
conn.close()

| participant_id   | participant_name   |   course_id | course_date   | prereq_to_complete   | prereq_completed   |
|:-----------------|:-------------------|------------:|:--------------|:---------------------|:-------------------|
| P002             | Bob Lee            |           3 | 2024-03-05    | 2 [1,2]              | 0 []               |
| P005             | Edward Goh         |           4 | 2024-03-15    | 1 [3]                | 0 []               |
| P007             | Grace Koh          |           3 | 2024-04-01    | 2 [1,2]              | 1 [1]              |
| P007             | Grace Koh          |           4 | 2024-05-25    | 1 [3]                | 0 []               |
| P007             | Grace Koh          |           5 | 2024-07-05    | 1 [4]                | 0 []               |
| P008             | Henry Teo          |           3 | 2024-03-25    | 2 [1,2]              | 0 []               |
| P008             | Henry Teo          |           4 | 2024-05-15    | 